# Huấn Luyện AI Sinh Đồ Họa Pixel Art Stardew Valley (Mini Diffusion Model)
Notebook này được thiết kế để chạy miễn phí trên **Google Colab (GPU NVIDIA Tesla T4 16GB VRAM)**.
Mục tiêu: Đào tạo một mạng nơ-ron khuếch tán mini (Lightweight Diffusion U-Net) từ 11.000 bản vẽ Stardew Valley để tự động sinh ra các nhân vật, chân dung và tài nguyên pixel art mới toanh 100% sáng tạo.

### Các Bước Thực Hiện:
1. Chọn menu: **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**
2. Chạy lần lượt các ô lệnh bên dưới (hoặc bấm **Runtime -> Run all**).
3. Tải file trọng số mô hình `th_stardew_generator.pt` về máy tính để tích hợp vào Web Studio!


### Bước 1: Kiểm Tra GPU NVIDIA T4 Và Cài Đặt Thư Viện


In [ ]:
!nvidia-smi

import torch
import torchvision
import os
import glob
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("CHÚ Ý: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU.")


### Bước 2: Tải Lên Và Giải Nén `stardew_dataset.zip`
Bạn có 2 cách đưa dữ liệu lên Google Colab:
- **Cách 1:** Bấm vào biểu tượng thư mục (Files) ở thanh bên trái của Colab rồi kéo thả file `stardew_dataset.zip` vào.
- **Cách 2:** Tải lên Google Drive rồi kết nối (Mount Drive).


In [ ]:
# Kiểm tra và giải nén stardew_dataset.zip
import zipfile

zip_path = "stardew_dataset.zip"

if not os.path.exists(zip_path):
    print("Đang chờ tải file stardew_dataset.zip...")
    from google.colab import files
    uploaded = files.upload()

if os.path.exists(zip_path):
    print("Đang giải nén dữ liệu ảnh...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("dataset")
    print("Giải nén hoàn tất!")
    
    # Đếm số lượng ảnh
    png_files = glob.glob("dataset/**/*.png", recursive=True)
    print(f"Tổng số ảnh PNG hợp lệ trong tập dữ liệu: {len(png_files)} tệp.")
else:
    print("LỖI: Không tìm thấy file stardew_dataset.zip!")


### Bước 3: Chuẩn Bị Dữ Liệu & Data Pipeline (PyTorch DataLoader)
Chuẩn hóa toàn bộ ảnh về kích thước chuẩn 64x64 pixel với thuật toán nội suy **Nearest-Neighbor** để bảo tồn độ sắc nét của từng hạt pixel art.


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class ThStardewPixelDataset(Dataset):
    def __init__(self, file_list, img_size=64):
        self.files = file_list
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.RandomHorizontalFlip(p=0.5), # Tăng cường dữ liệu (lật gương nhân vật)
            transforms.ToTensor(), # Chuyển về Tensor [0, 1]
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Chuẩn hóa [-1, 1]
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        try:
            # Mở ảnh và chuyển sang RGBA / RGB
            img = Image.open(path).convert('RGB')
            return self.transform(img)
        except Exception as e:
            # Nếu gặp ảnh lỗi, trả về ảnh ngẫu nhiên khác
            return self.__getitem__((idx + 1) % len(self.files))

# Khởi tạo DataLoader tối ưu cho GPU Tesla T4
IMG_SIZE = 64
BATCH_SIZE = 64 # T4 có 16GB VRAM, batch size 64 hoặc 128 chạy cực kỳ mượt mà!

dataset = ThStardewPixelDataset(png_files, img_size=IMG_SIZE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f"Khởi tạo DataLoader thành công: {len(dataset)} ảnh | {len(dataloader)} batches mỗi epoch.")

# Xem thử 4 ảnh mẫu
sample_batch = next(iter(dataloader))
fig, axs = plt.subplots(1, 4, figsize=(10, 3))
for i in range(4):
    img_disp = (sample_batch[i].permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    axs[i].imshow(img_disp, interpolation='nearest')
    axs[i].axis('off')
    axs[i].set_title(f"Mẫu {i+1}")
plt.show()


### Bước 4: Xây Dựng Mạng Nơ-ron Khuếch Tán (Lightweight Pixel Diffusion U-Net)
Mô hình U-Net được thiết kế đặc thù cho Pixel Art với các khối tích chập Residual và cơ chế Attention ở tầng trung tâm, tích hợp bộ lịch trình khuếch tán DDPM.


In [ ]:
import torch.nn as nn
import math

class ThSinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class ThBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, up=False):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2*in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()

    def forward(self, x, t):
        h = self.bnorm1(self.relu(self.conv1(x)))
        time_emb = self.relu(self.time_mlp(t))
        time_emb = time_emb[(..., ) + (None, ) * 2]
        h = h + time_emb
        h = self.bnorm2(self.relu(self.conv2(h)))
        return self.transform(h)

class ThSimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        image_channels = 3
        down_channels = (64, 128, 256)
        up_channels = (256, 128, 64)
        out_dim = 3
        time_emb_dim = 64

        self.time_mlp = nn.Sequential(
            ThSinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )
        self.conv0 = nn.Conv2d(image_channels, down_channels[0], 3, padding=1)
        self.downs = nn.ModuleList([ThBlock(down_channels[i], down_channels[i+1], time_emb_dim) for i in range(len(down_channels)-1)])
        self.ups = nn.ModuleList([ThBlock(up_channels[i], up_channels[i+1], time_emb_dim, up=True) for i in range(len(up_channels)-1)])
        self.output = nn.Conv2d(up_channels[-1], out_dim, 1)

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x = self.conv0(x)
        residual_inputs = []
        for down in self.downs:
            x = down(x, t)
            residual_inputs.append(x)
        for up in self.ups:
            residual_x = residual_inputs.pop()
            x = torch.cat((x, residual_x), dim=1)
            x = up(x, t)
        return self.output(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ThSimpleUNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Kiến trúc U-Net đã sẵn sàng trên: {device}")
print(f"Tổng số tham số: {total_params:,} tham số (~{total_params*4 / (1024**2):.1f} MB bộ nhớ).")


### Bước 5: Lịch Trình Khử Nhiễu (DDPM Schedule) & Quá Trình Huấn Luyện (Training Loop)
Áp dụng công nghệ Mixed Precision (FP16) trên lõi Tensor Core của GPU Tesla T4 để đạt tốc độ tối đa.


In [ ]:
TIMESTEPS = 300 # 300 bước khuếch tán (tối ưu hóa cho tốc độ và chất lượng pixel)

def th_linear_beta_schedule(timesteps):
    beta_start = 0.0001
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, timesteps)

betas = th_linear_beta_schedule(timesteps=TIMESTEPS).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, axis=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0]).to(device), alphas_cumprod[:-1]])
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - alphas_cumprod)

def th_q_sample(x_start, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x_start)
    sqrt_alphas_cumprod_t = sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
    sqrt_one_minus_alphas_cumprod_t = sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
    return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise

# Cấu hình tối ưu hóa
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
criterion = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler() # Tăng tốc bằng FP16

EPOCHS = 35 # 35 epochs mất khoảng 15-20 phút trên Tesla T4

print(f"BẮT ĐẦU HUẤN LUYỆN TRÊN GPU {torch.cuda.get_device_name(0)}...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}")
    
    for batch in pbar:
        optimizer.zero_grad()
        x_0 = batch.to(device)
        t = torch.randint(0, TIMESTEPS, (x_0.shape[0],), device=device).long()
        noise = torch.randn_like(x_0)
        x_noisy = th_q_sample(x_0, t, noise)
        
        with torch.cuda.amp.autocast():
            noise_pred = model(x_noisy, t)
            loss = criterion(noise_pred, noise)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
        
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch} hoàn thành | Average Loss: {avg_loss:.5f}")

print("HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")


### Bước 6: Thử Nghiệm Sinh Ảnh Pixel Art Mới Toanh (Sampling / Generation)
Mô hình sẽ bắt đầu từ hạt nhiễu ngẫu nhiên thuần túy và khử nhiễu dần qua 300 bước để tạo ra các tác phẩm pixel art hoàn toàn mới!


In [ ]:
@torch.no_grad()
def th_generate_pixel_art(model, num_samples=8):
    model.eval()
    img = torch.randn((num_samples, 3, IMG_SIZE, IMG_SIZE), device=device)
    
    print("Đang khử nhiễu sinh ảnh pixel art mới...")
    for i in tqdm(reversed(range(0, TIMESTEPS)), total=TIMESTEPS, desc="Denoising"):
        t = torch.full((num_samples,), i, device=device, dtype=torch.long)
        with torch.cuda.amp.autocast():
            predicted_noise = model(img, t)
            
        beta = betas[i]
        alpha = alphas[i]
        alpha_hat = alphas_cumprod[i]
        
        if i > 0:
            noise = torch.randn_like(img)
        else:
            noise = torch.zeros_like(img)
            
        img = (1 / torch.sqrt(alpha)) * (img - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * predicted_noise) + torch.sqrt(beta) * noise
        
    img = (img * 0.5 + 0.5).clamp(0, 1)
    return img

generated_images = th_generate_pixel_art(model, num_samples=8)

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for idx in range(8):
    row, col = idx // 4, idx % 4
    img_np = generated_images[idx].permute(1, 2, 0).cpu().numpy()
    axs[row, col].imshow(img_np, interpolation='nearest')
    axs[row, col].axis('off')
    axs[row, col].set_title(f"Pixel Sinh Ra {idx+1}")
plt.tight_layout()
plt.show()


### Bước 7: Lưu Trọng Số Mô Hình Và Tải Về Máy Tính
File mô hình hoàn chỉnh sẽ được lưu và tự động tải trực tiếp về thư mục `Downloads` của máy tính bạn.


In [ ]:
# Lưu trọng số PyTorch và ONNX
model_path = "th_stardew_generator.pt"
torch.save(model.state_dict(), model_path)
print(f"Đã lưu mô hình tại: {model_path} ({os.path.getsize(model_path) / (1024**2):.2f} MB)")

# Tải trực tiếp file về máy
from google.colab import files
files.download(model_path)
print("File đã được gửi về trình duyệt của bạn!")
